In [ ]:
from google.colab import files
uploaded = files.upload()

Saving df3_vendedores_categorias.csv to df3_vendedores_categorias (1).csv


In [ ]:
import pandas as pd
df = pd.read_csv('df3_vendedores_categorias.csv')
df.head()

,vendedor,estado,categoria,ventas,ventas_totales_vendedor,num_categorias,ranking_categoria,porcentaje_vendedor
0,6560211a19b47992c3666cc44a7e94c0,SP,relogios_presentes,1628,2033,7,1,80.08
1,6560211a19b47992c3666cc44a7e94c0,SP,fashion_bolsas_e_acessorios,340,2033,7,2,16.72
2,6560211a19b47992c3666cc44a7e94c0,SP,audio,32,2033,7,3,1.57
3,6560211a19b47992c3666cc44a7e94c0,SP,perfumaria,13,2033,7,4,0.64
4,6560211a19b47992c3666cc44a7e94c0,SP,informatica_acessorios,12,2033,7,5,0.59


Hacemos un diagnostico previo a la limpieza

In [ ]:
print("Filas y columnas:")
print(df.shape)

print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos:")
print(df.isnull().sum())

print("\nDuplicados:")
print(df.duplicated().sum())

Filas y columnas:
(6359, 8)

Tipos de datos:
vendedor                    object
estado                      object
categoria                   object
ventas                       int64
ventas_totales_vendedor      int64
num_categorias               int64
ranking_categoria            int64
porcentaje_vendedor        float64
dtype: object

Valores nulos:
vendedor                   0
estado                     0
categoria                  0
ventas                     0
ventas_totales_vendedor    0
num_categorias             0
ranking_categoria          0
porcentaje_vendedor        0
dtype: int64

Duplicados:
0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6359 entries, 0 to 6358
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   vendedor                 6359 non-null   object 
 1   estado                   6359 non-null   object 
 2   categoria                6359 non-null   object 
 3   ventas                   6359 non-null   int64  
 4   ventas_totales_vendedor  6359 non-null   int64  
 5   num_categorias           6359 non-null   int64  
 6   ranking_categoria        6359 non-null   int64  
 7   porcentaje_vendedor      6359 non-null   float64
dtypes: float64(1), int64(4), object(3)
memory usage: 397.6+ KB


In [ ]:
df.isnull().sum()

,0
vendedor,0
estado,0
categoria,0
ventas,0
ventas_totales_vendedor,0
num_categorias,0
ranking_categoria,0
porcentaje_vendedor,0


No hay valores nulos ni duplicados pues los hemos corregidos en la limpieza previa en SQL.

In [ ]:
df['categoria'].nunique()

73

In [ ]:
sorted(df['categoria'].unique())

Unificamos estas categorías

In [ ]:
df['categoria'] = df['categoria'].replace({
    'casa_conforto_2': 'casa_conforto',
    'eletrodomesticos_2': 'eletrodomesticos'
})


In [ ]:
df['categoria'].nunique()

71

Reajustamos la tabla

In [ ]:
df = (
    df.groupby(['vendedor', 'estado', 'categoria'], as_index=False)
      .agg({'ventas': 'sum'})
)

In [ ]:
df['ventas_totales_vendedor'] = df.groupby('vendedor')['ventas'].transform('sum')

df['num_categorias'] = df.groupby('vendedor')['categoria'].transform('count')

df['ranking_categoria'] = (
    df.groupby('vendedor')['ventas']
      .rank(method='first', ascending=False)
      .astype(int)
)

df['porcentaje_vendedor'] = round(
    100 * df['ventas'] / df['ventas_totales_vendedor'],
    2
)

In [ ]:
df[df['vendedor'] == '4a3ca9315b744ce9f8e9374361493884']

,vendedor,estado,categoria,ventas,ventas_totales_vendedor,num_categorias,ranking_categoria,porcentaje_vendedor
1812,4a3ca9315b744ce9f8e9374361493884,SP,bebes,5,1987,6,5,0.25
1813,4a3ca9315b744ce9f8e9374361493884,SP,brinquedos,55,1987,6,4,2.77
1814,4a3ca9315b744ce9f8e9374361493884,SP,cama_mesa_banho,1572,1987,6,1,79.11
1815,4a3ca9315b744ce9f8e9374361493884,SP,casa_conforto,269,1987,6,2,13.54
1816,4a3ca9315b744ce9f8e9374361493884,SP,cool_stuff,1,1987,6,6,0.05
1817,4a3ca9315b744ce9f8e9374361493884,SP,moveis_decoracao,85,1987,6,3,4.28


In [ ]:
df = df.sort_values(
    ['ventas_totales_vendedor', 'vendedor', 'ventas'],
    ascending=[False, True, False]
).reset_index(drop=True)